In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch
import pickle
import os

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# Load the dataset
def load_data():
    movies = pd.read_csv('data/movies.csv')
    return movies

In [ ]:
# Preprocess the data
def preprocess_data(movies):
    # Fill missing descriptions with an empty string
    movies['description'] = movies['description'].fillna('')
    return movies

In [ ]:
# Generate TF-IDF embeddings
def generate_tfidf_embeddings(movies):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(movies['description'])
    return tfidf, tfidf_matrix

In [ ]:
# Generate Transformer embeddings (e.g., BERT)
def generate_transformer_embeddings(movies):
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    model = AutoModel.from_pretrained('bert-base-uncased')
    
    embeddings = []
    for desc in movies['description']:
        inputs = tokenizer(desc, return_tensors='pt', truncation=True, padding=True)
        with torch.no_grad():
            output = model(**inputs)
        embeddings.append(output.last_hidden_state.mean(dim=1).numpy())
    
    return np.vstack(embeddings)

In [ ]:
# Save model and vectorizer to pickle files
def save_artifacts(tfidf, transformer_embeddings):
    os.makedirs('models', exist_ok=True)
    with open('models/tfidf_vectorizer.pickle', 'wb') as f:
        pickle.dump(tfidf, f)
    with open('models/transformer_model.pickle', 'wb') as f:
        pickle.dump(transformer_embeddings, f)

In [ ]:

# Load model and vectorizer from pickle files
def load_artifacts():
    with open('models/tfidf_vectorizer.pickle', 'rb') as f:
        tfidf = pickle.load(f)
    with open('models/transformer_model.pickle', 'rb') as f:
        transformer_embeddings = pickle.load(f)
    return tfidf, transformer_embeddings

In [ ]:
# Recommend movies based on cosine similarity
def recommend_movies(movie_title, movies, embeddings, top_n=5):
    # Find the index of the movie
    movie_index = movies[movies['title'] == movie_title].index[0]
    
    # Compute cosine similarity
    cosine_sim = cosine_similarity(embeddings[movie_index].reshape(1, -1), embeddings)
    
    # Get top N similar movies
    similar_movies_indices = cosine_sim.argsort()[0][-top_n-1:-1][::-1]
    similar_movies = movies.iloc[similar_movies_indices]
    
    return similar_movies

In [ ]:
# Main function
def main():
    # Load and preprocess data
    movies = load_data()
    movies = preprocess_data(movies)
    
    # Check if artifacts exist
    if not (os.path.exists('models/tfidf_vectorizer.pickle') and os.path.exists('models/transformer_model.pickle')):
        print("Generating embeddings...")
        tfidf, tfidf_matrix = generate_tfidf_embeddings(movies)
        transformer_embeddings = generate_transformer_embeddings(movies)
        save_artifacts(tfidf, transformer_embeddings)
    else:
        print("Loading embeddings...")
        tfidf, transformer_embeddings = load_artifacts()
    
    # Example: Recommend movies similar to "Toy Story"
    movie_title = "Toy Story"
    recommendations = recommend_movies(movie_title, movies, transformer_embeddings)
    print(f"\nTop 5 Recommendations for '{movie_title}':")
    print(recommendations[['title', 'description']])

if __name__ == "__main__":
    main()